# Conditional DDPM (V2 - Step 4) - RESUME notebook

Standalone re-run of the **vanilla conditional DDPM** step using the checkpoints already saved on Drive. Skips SHAP / IG / ensemble / classifier-training cells so nothing heavy is recomputed or re-uploaded.

**Run order:** all cells top-to-bottom. The DDPM training cell is resume-guarded (`if DDPM_CKPT.exists(): skip`), so with `ddpm_fundus_v1.pt` on Drive it loads instead of retraining.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm captum shap grad-cam scikit-image opencv-python-headless tabulate
!pip install -q diffusers transformers accelerate

In [ ]:
from pathlib import Path
import zipfile

# ----- Drive (input + persistent output) -----
DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
THESIS_ZIP      = DRIVE_ROOT / 'Thesis.zip'          # single outer zip
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'

In [ ]:
LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
EYEQ_DIR     = LOCAL_ROOT / 'eyeq'
PRISTINE_DIR = LOCAL_ROOT / 'pristine'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
ENHANCED_DIR = LOCAL_ROOT / 'enhanced'

PHASE_DIRS = {
    'phase1_data_engineering':   RESULTS_ROOT / 'phase1_data_engineering',
    'phase2_model_benchmarking': RESULTS_ROOT / 'phase2_model_benchmarking',
    'phase3_xai_benchmark':      RESULTS_ROOT / 'phase3_xai_benchmark',
    'phase4_genai_enhancement':  RESULTS_ROOT / 'phase4_genai_enhancement',
    'phase5_quality_ensemble':   RESULTS_ROOT / 'phase5_quality_ensemble',
}
PHASE_SUBDIRS = ('metrics', 'plots', 'samples', 'logs')

In [ ]:
for p in [LOCAL_ROOT, PRISTINE_DIR, DEGRADED_DIR, ENHANCED_DIR,
          RESULTS_ROOT, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for phase_dir in PHASE_DIRS.values():
    for sub in PHASE_SUBDIRS:
        (phase_dir / sub).mkdir(parents=True, exist_ok=True)

print('Drive zip   :', THESIS_ZIP, '(exists:', THESIS_ZIP.exists(), ')')
print('Drive results tree:')
for k, v in PHASE_DIRS.items():
    print(f'  {k:35s} -> {v}')

In [ ]:
# Idempotent extraction — skips automatically if already done in this Colab session.
# Saves 5-15 minutes per reconnect. Force re-extract: smart_extract(force=True).
import zipfile, shutil, time

EXTRACT_MARK = LOCAL_ROOT / ".extracted_ok"        # sentinel file


def _has_data():
    if not EXTRACT_MARK.exists():
        return False
    has_aptos = any(APTOS_DIR.rglob("*.png")) or any(APTOS_DIR.rglob("*.jpg"))
    has_eyeq  = any(EYEQ_DIR.rglob("*.png"))  or any(EYEQ_DIR.rglob("*.jpg"))
    return has_aptos and has_eyeq


def _find_dir(root: Path, keywords):
    for p in root.rglob("*"):
        if p.is_dir() and any(k in p.name.lower() for k in keywords):
            return p
    return None


def smart_extract(force: bool = False):
    """Extract Thesis.zip -> /content/data, then recursively unpack inner zips.
    Skips if a previous extraction is still in place."""
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    if _has_data() and not force:
        n_aptos = sum(1 for _ in APTOS_DIR.rglob("*.*"))
        n_eyeq  = sum(1 for _ in EYEQ_DIR.rglob("*.*"))
        print(f"[skip] data already extracted (aptos={n_aptos}, eyeq={n_eyeq})")
        return

    if not THESIS_ZIP.exists():
        raise FileNotFoundError(f"Missing zip: {THESIS_ZIP}")

    t0 = time.time()
    print(f"[extract] {THESIS_ZIP.name} -> {LOCAL_ROOT}")
    with zipfile.ZipFile(THESIS_ZIP) as z:
        z.extractall(LOCAL_ROOT)

    while True:
        inners = list(LOCAL_ROOT.rglob("*.zip"))
        if not inners:
            break
        for iz in inners:
            target = iz.parent / iz.stem
            target.mkdir(exist_ok=True)
            print(f"  inner zip: {iz.name} -> {target}")
            with zipfile.ZipFile(iz) as z:
                z.extractall(target)
            iz.unlink()

    # Symlink canonical paths
    aptos_src = _find_dir(LOCAL_ROOT, ["aptos"])
    eyeq_src  = _find_dir(LOCAL_ROOT, ["eyeq", "eye_q", "eye-q"])
    print("Detected APTOS dir:", aptos_src)
    print("Detected EyeQ  dir:", eyeq_src)
    for canonical, real in [(APTOS_DIR, aptos_src), (EYEQ_DIR, eyeq_src)]:
        if real is None:
            print(f"[warn] {canonical.name} not found — check Thesis.zip contents")
            continue
        if canonical.exists() and canonical.resolve() != real.resolve():
            if canonical.is_dir() and not canonical.is_symlink():
                shutil.rmtree(canonical)
            else:
                canonical.unlink()
        if not canonical.exists():
            canonical.symlink_to(real, target_is_directory=True)
        print(f"  {canonical} -> {real}")

    EXTRACT_MARK.touch()
    print(f"[extract] done in {time.time()-t0:.1f}s")


smart_extract()         # subsequent calls in this session are instant


In [ ]:
import torch
PHASE_DIRS = {
    'phase1_data_engineering':   RESULTS_ROOT / 'phase1_data_engineering',
    'phase2_model_benchmarking': RESULTS_ROOT / 'phase2_model_benchmarking',
    'phase3_xai_benchmark':      RESULTS_ROOT / 'phase3_xai_benchmark',
    'phase4_genai_enhancement':  RESULTS_ROOT / 'phase4_genai_enhancement',
    'phase5_quality_ensemble':   RESULTS_ROOT / 'phase5_quality_ensemble',
}
for p in [DEGRADED_DIR, ENHANCED_DIR, RESULTS_ROOT, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for d in PHASE_DIRS.values():
    for sub in ('metrics', 'plots', 'samples', 'logs'):
        (d / sub).mkdir(parents=True, exist_ok=True)

# Auto-detect APTOS train CSV + image dir (case-insensitive)
APTOS_CSV    = next(p for p in APTOS_DIR.rglob('*.csv') if 'train' in p.name.lower())
APTOS_IMAGES = next(p for p in APTOS_DIR.rglob('*train_images*') if p.is_dir())
EYEQ_CSV     = next(EYEQ_DIR.rglob('*.csv'), None)

# Common constants
NUM_CLASSES = 5
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
IMAGE_SIZE  = 224
SEED        = 42
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15

MODEL_NAMES = ('resnet50', 'efficientnet_b3', 'vit_base')
TIMM_NAMES  = {'resnet50': 'resnet50',
               'efficientnet_b3': 'efficientnet_b3',
               'vit_base': 'vit_base_patch16_224'}
TRAIN_CFG = dict(batch_size=32, num_workers=2, epochs=15,
                 lr=3e-4, weight_decay=1e-4, label_smoothing=0.05,
                 mixed_precision=True)

DEGRADATION_TYPES  = ('blur', 'exposure', 'noise')
DEGRADATION_LEVELS = ('low', 'mid', 'high')
DEGRADATION_PARAMS = {
    'blur':     {'low': 2.0,  'mid': 5.0,  'high': 9.0},
    'exposure': {'low': 0.7,  'mid': 0.4,  'high': 0.2},
    'noise':    {'low': 0.02, 'mid': 0.06, 'high': 0.12},
}
EYEQ_QUALITY_LABELS = {0: 'good', 1: 'usable', 2: 'reject'}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('APTOS csv   :', APTOS_CSV)
print('APTOS images:', APTOS_IMAGES)
print('EyeQ csv    :', EYEQ_CSV)
print('Device      :', DEVICE)

In [ ]:
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms as T
import timm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from scipy.stats import spearmanr
from skimage.segmentation import slic

TFM_TRAIN = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        T.RandomHorizontalFlip(),
                        T.RandomRotation(15),
                        T.ColorJitter(0.1, 0.1, 0.1),
                        T.ToTensor(),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
TFM_EVAL  = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        T.ToTensor(),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def load_tensor(path: Path) -> torch.Tensor:
    return TFM_EVAL(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)

In [ ]:
class APTOSDataset(Dataset):
    """Resolves images via IMAGE_INDEX so any extension / nested layout works."""
    def __init__(self, csv_path, images_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.images_dir = Path(images_dir)   # kept for API compatibility, not used
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(resolve_image(row['id_code'])).convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

class FolderDataset(Dataset):
    """Reads `<root>/manifest.csv` (cols: id_code, diagnosis, rel_path)."""
    def __init__(self, root, transform=TFM_EVAL):
        self.root = Path(root); self.df = pd.read_csv(self.root / 'manifest.csv')
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row['rel_path']).convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

In [ ]:
# === Improved training infrastructure ===
import math
from contextlib import contextmanager
from collections import Counter
from torch.utils.data import WeightedRandomSampler, Subset
from sklearn.model_selection import train_test_split

# 1) STRATIFIED SPLIT — keeps class proportions consistent across train/val/test
def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx = np.arange(len(labels)); labels = np.array(labels)
    tr_idx, te_idx = train_test_split(idx, test_size=test_frac, stratify=labels, random_state=seed)
    rel_val = val_frac / (1 - test_frac)
    tr_idx, va_idx = train_test_split(tr_idx, test_size=rel_val,
                                       stratify=labels[tr_idx], random_state=seed)
    return tr_idx.tolist(), va_idx.tolist(), te_idx.tolist()

# 2) CLASS WEIGHTS + BALANCED SAMPLER
def class_weights_tensor(labels, num_classes=NUM_CLASSES):
    cnt = Counter(labels); n = sum(cnt.values())
    return torch.tensor([n / (num_classes * max(cnt.get(i, 0), 1)) for i in range(num_classes)],
                         dtype=torch.float)

def balanced_sampler(labels):
    cnt = Counter(labels)
    w = [1.0 / cnt[l] for l in labels]
    return WeightedRandomSampler(w, num_samples=len(w), replacement=True)

# 3) FOCAL LOSS (handles soft targets from MixUp + per-class weights)
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__(); self.gamma, self.alpha, self.ls = gamma, alpha, label_smoothing
    def forward(self, logits, targets):
        if targets.dim() > 1:                       # soft labels (MixUp)
            log_p = F.log_softmax(logits, dim=-1)
            return -(targets * log_p).sum(-1).mean()
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                              reduction='none', label_smoothing=self.ls)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

# 4) EMA — exponential moving average of weights, +0.5–1% reliably
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
    def update(self, model):
        with torch.no_grad():
            for n, p in model.named_parameters():
                if p.requires_grad:
                    self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    @contextmanager
    def applied(self, model):
        backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.shadow[n])
        try: yield
        finally:
            for n, p in model.named_parameters():
                if p.requires_grad: p.data.copy_(backup[n])

# 5) STRONG TRAIN AUGMENTATION (medical-imaging safe — no extreme hue shifts)
TFM_TRAIN_STRONG = T.Compose([
    T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(25),
    T.RandAugment(num_ops=2, magnitude=7),
    T.ColorJitter(0.15, 0.15, 0.15, 0.03),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.25),
])

# 6) MIXUP / CUTMIX (timm's implementation)
from timm.data import Mixup

# 7) LAYER-WISE LR DECAY (LRD) — critical for ViT fine-tuning
def vit_lrd_param_groups(model, base_lr, decay=0.75, weight_decay=5e-2):
    n_layers = len(model.blocks) + 1
    groups = []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith('head'):                                  depth = n_layers
        elif n.startswith('blocks'):                              depth = int(n.split('.')[1])
        elif any(n.startswith(s) for s in
                 ('patch_embed', 'cls_token', 'pos_embed', 'norm')): depth = 0
        else:                                                     depth = 0
        lr_scale = decay ** (n_layers - depth)
        groups.append({'params': p, 'lr': base_lr * lr_scale,
                       'weight_decay': 0.0 if p.ndim <= 1 else weight_decay})
    return groups

# 8) WARMUP + COSINE LR SCHEDULE
def warmup_cosine(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    def lr_lambda(step):
        epoch = step / max(steps_per_epoch, 1)
        if epoch < warmup_epochs:
            return epoch / max(warmup_epochs, 1)
        prog = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# 9) PER-ARCHITECTURE HYPERPARAMETERS
HPARAMS = {
    'resnet50':        dict(lr=3e-4, wd=1e-4, epochs=25, mixup=0.2, ls=0.1, gamma=1.5, warmup=3, llrd=None),
    'efficientnet_b3': dict(lr=2e-4, wd=1e-5, epochs=25, mixup=0.2, ls=0.1, gamma=2.0, warmup=3, llrd=None),
    'vit_base':        dict(lr=5e-5, wd=5e-2, epochs=30, mixup=0.4, ls=0.1, gamma=2.0, warmup=5, llrd=0.75),
}

print('Improved training infrastructure loaded.')

In [ ]:
P1 = PHASE_DIRS['phase1_data_engineering']

# 1.1 V2 PATCH — phantom EyeQ filter removed.
# The original filter_pristine() tried to join EyeQ quality labels onto APTOS
# but the join key never matched in any observed Colab run — the routine
# silently fell back to "use all APTOS rows". To keep the methodology honest
# we now explicitly use the full APTOS dataset and document "APTOS only" in
# the dissertation. EyeQ is still loaded separately for the Phase 5 quality
# classifier training (Q_CKPT).
def filter_pristine(aptos_csv, eyeq_csv=None, out_csv=None, **kwargs):
    """V2 patch: always returns the full APTOS dataset."""
    aptos = pd.read_csv(aptos_csv)
    if out_csv is not None:
        aptos.to_csv(out_csv, index=False)
    return aptos

PRISTINE_CSV = P1 / 'metrics' / 'pristine_split.csv'
pristine_df = filter_pristine(APTOS_CSV, EYEQ_CSV, PRISTINE_CSV)
print(f'Using full APTOS dataset: {len(pristine_df)} images')
pristine_df.head()


In [ ]:
# Diagnose the actual APTOS image layout
print('APTOS_IMAGES :', APTOS_IMAGES)
print('Exists?       :', APTOS_IMAGES.exists())
all_files = list(APTOS_IMAGES.rglob('*'))
img_files = [p for p in all_files if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
print(f'Total files under APTOS_IMAGES (any depth): {len(all_files)}')
print(f'Image files found                          : {len(img_files)}')
if img_files:
    print('Sample image paths:')
    for p in img_files[:5]:
        print(' ', p.relative_to(APTOS_IMAGES))
print('\nSample id_code from CSV:', pristine_df["id_code"].iloc[0])
print('Sample image stem      :', img_files[0].stem if img_files else '(none)')

In [ ]:
# Build a stem -> path index so we don't care about extension or subfolder depth.
# Searches all of APTOS_DIR (not just APTOS_IMAGES) in case auto-detection picked the wrong folder.
print('Indexing APTOS images...')
IMAGE_INDEX = {}
for p in APTOS_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.tif', '.tiff'):
        IMAGE_INDEX[p.stem] = p
print(f'Indexed {len(IMAGE_INDEX)} images.')

# Sanity check against the CSV
csv_ids = pristine_df['id_code'].astype(str).tolist()
matched = [i for i in csv_ids if i in IMAGE_INDEX]
missing = [i for i in csv_ids if i not in IMAGE_INDEX]
print(f'CSV ids matched: {len(matched)}/{len(csv_ids)}')
if missing:
    print(f'Example missing id: {missing[0]}')
    print('Example indexed stems:', list(IMAGE_INDEX)[:3])
    raise SystemExit('CSV ids do not match image filenames — inspect the diagnostic output above.')

def resolve_image(id_code):
    """Get the actual image path for an APTOS id_code, regardless of extension or folder."""
    p = IMAGE_INDEX.get(str(id_code))
    if p is None:
        raise FileNotFoundError(f'No image indexed for id_code={id_code!r}')
    return p

In [ ]:
# 1.3 Synthetic degradation primitives
def _np(img):  return np.array(img.convert('RGB'))
def _pil(arr): return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def gaussian_blur(img, sigma):
    k = max(3, int(2 * round(3 * sigma) + 1))
    return _pil(cv2.GaussianBlur(_np(img), (k, k), sigmaX=sigma, sigmaY=sigma))

def exposure_shift(img, gain): return _pil(_np(img).astype(np.float32) * gain)
def gaussian_noise(img, std_frac):
    arr = _np(img).astype(np.float32)
    return _pil(arr + np.random.normal(0, std_frac * 255, arr.shape))

DEGRADERS = {'blur': gaussian_blur, 'exposure': exposure_shift, 'noise': gaussian_noise}
def apply_degradation(img, kind, level):
    return DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])

## Pre-flight check + DDPM step

In [ ]:
# --- pre-flight: confirm the DDPM checkpoint is on Drive before we rely on it ---
import random
import torchvision.transforms as T
_ddpm_ckpt = CHECKPOINTS_DIR / 'ddpm_fundus_v1.pt'
assert CHECKPOINTS_DIR.exists(), f'Drive checkpoints dir missing: {CHECKPOINTS_DIR}'
print('checkpoints dir :', CHECKPOINTS_DIR)
print('ddpm checkpoint :', _ddpm_ckpt, '->', 'FOUND' if _ddpm_ckpt.exists() else 'MISSING')
print('PRISTINE_CSV    :', PRISTINE_CSV, '->', 'FOUND' if PRISTINE_CSV.exists() else 'MISSING')
if not _ddpm_ckpt.exists():
    print('\n[warning] checkpoint missing -> the next cell WILL train from scratch (~hours).')


In [ ]:
# === V2 PATCH (Step 4): conditional vanilla DDPM ===
import math
import torch.nn as nn
import torch.nn.functional as F

DDPM_CKPT = CHECKPOINTS_DIR / 'ddpm_fundus_v1.pt'
DDPM_SIZE = 256


class _SinusoidalPE(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000.0) / max(half - 1, 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class _DDPMUNet(nn.Module):
    """U-Net conditioned on (timestep, degraded image).
    Input  : concat(noisy_clean, degraded) -> 6 channels.
    Output : predicted noise -> 3 channels.
    """
    def __init__(self, in_ch=6, out_ch=3, base_ch=64, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            _SinusoidalPE(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim),
        )
        def _block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.GroupNorm(8, out_c),
                nn.GELU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.GroupNorm(8, out_c),
                nn.GELU(),
            )
        self.enc1 = _block(in_ch, base_ch)
        self.enc2 = _block(base_ch, base_ch * 2)
        self.enc3 = _block(base_ch * 2, base_ch * 4)
        self.mid  = _block(base_ch * 4, base_ch * 4)
        self.film = nn.Linear(time_dim, base_ch * 4 * 2)
        self.dec3 = _block(base_ch * 8, base_ch * 2)
        self.dec2 = _block(base_ch * 4, base_ch)
        self.dec1 = _block(base_ch * 2, base_ch)
        self.final = nn.Conv2d(base_ch, out_ch, 1)
        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m  = self.mid(self.pool(e3))
        scale, shift = self.film(t_emb).unsqueeze(-1).unsqueeze(-1).chunk(2, dim=1)
        m = m * (1 + scale) + shift
        d3 = self.dec3(torch.cat([self.up(m),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        return self.final(d1)


class FundusDDPM:
    """Standard DDPM (Ho et al. 2020) conditioned on the degraded image."""
    def __init__(self, model, T=1000, beta_start=1e-4, beta_end=0.02, device='cuda'):
        self.model = model
        self.T = T
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1 - self.alpha_cumprod)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_acp   = self.sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
        sqrt_omacp = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
        return sqrt_acp * x0 + sqrt_omacp * noise, noise

    def train_step(self, clean, degraded, optimizer, scaler=None):
        B = clean.size(0)
        t = torch.randint(0, self.T, (B,), device=self.device)
        noise = torch.randn_like(clean)
        noisy_clean, _ = self.q_sample(clean, t, noise)
        inp = torch.cat([noisy_clean, degraded], dim=1)   # (B, 6, H, W)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast(enabled=True):
                pred = self.model(inp, t)
                loss = F.mse_loss(pred, noise)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        else:
            pred = self.model(inp, t)
            loss = F.mse_loss(pred, noise)
            loss.backward(); optimizer.step()
        return loss.item()

    @torch.no_grad()
    def restore(self, degraded, n_steps=50):
        B = degraded.size(0)
        x = torch.randn(B, 3, degraded.size(2), degraded.size(3), device=self.device)
        step_size = max(self.T // n_steps, 1)
        timesteps = list(range(self.T - 1, 0, -step_size))
        for t_val in timesteps:
            t = torch.full((B,), t_val, device=self.device, dtype=torch.long)
            inp = torch.cat([x, degraded], dim=1)
            pred_noise = self.model(inp, t)
            alpha = self.alphas[t_val]
            alpha_cumprod = self.alpha_cumprod[t_val]
            beta = self.betas[t_val]
            x = (1 / torch.sqrt(alpha)) * (
                x - (beta / torch.sqrt(1 - alpha_cumprod)) * pred_noise
            )
            if t_val > 1:
                x = x + torch.sqrt(beta) * torch.randn_like(x)
        return x.clamp(-1, 1)


class _DDPMDataset(Dataset):
    def __init__(self, df, n_per=4):
        self.df = df.reset_index(drop=True); self.n_per = n_per
        self.tfm = T.Compose([T.Resize((DDPM_SIZE, DDPM_SIZE)),
                              T.ToTensor(),
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    def __len__(self): return len(self.df) * self.n_per
    def __getitem__(self, i):
        row = self.df.iloc[i // self.n_per]
        try:
            img = Image.open(resolve_image(row['id_code'])).convert('RGB')
        except FileNotFoundError:
            img = Image.new('RGB', (DDPM_SIZE, DDPM_SIZE))
        img = img.resize((DDPM_SIZE, DDPM_SIZE), Image.BILINEAR)
        kind  = random.choice(['blur', 'exposure', 'noise'])
        level = random.choice(['low', 'mid', 'high'])
        deg   = DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])
        return self.tfm(deg), self.tfm(img)


def train_ddpm(epochs=15, batch_size=8, lr=1e-4, T=1000):
    """Resume-guarded conditional DDPM training (~3-6 hr on A100)."""
    if DDPM_CKPT.exists():
        print(f"[skip] {DDPM_CKPT.name} already on Drive — not retraining.")
        return
    print(f"Training conditional DDPM ({epochs} epochs, T={T}, batch={batch_size}, lr={lr})...")
    df  = pd.read_csv(PRISTINE_CSV)
    ds  = _DDPMDataset(df, n_per=4)
    dl  = DataLoader(ds, batch_size=batch_size, shuffle=True,
                     num_workers=2, pin_memory=True)
    net = _DDPMUNet().to(DEVICE)
    ddpm = FundusDDPM(net, T=T, device=DEVICE)
    opt  = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-5)
    scaler = GradScaler(enabled=True)
    for ep in range(epochs):
        net.train()
        running, n = 0.0, 0
        pbar = tqdm(dl, desc=f"ddpm ep {ep+1}/{epochs}", leave=False)
        for deg, clean in pbar:
            deg   = deg.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)
            loss = ddpm.train_step(clean, deg, opt, scaler=scaler)
            running += loss * deg.size(0); n += deg.size(0)
            pbar.set_postfix(MSE=f"{running/n:.4f}")
        print(f"  ep {ep+1}: MSE={running/n:.4f}")
    torch.save({'state_dict': net.state_dict(), 'T': T}, DDPM_CKPT)
    print(f"Saved -> {DDPM_CKPT}")


train_ddpm(epochs=15, batch_size=8, lr=1e-4, T=1000)


# ---- Inference ----
_ddpm_net = None
_ddpm     = None
def _load_ddpm(T=1000):
    global _ddpm_net, _ddpm
    if _ddpm is not None:
        return _ddpm
    _ddpm_net = _DDPMUNet().to(DEVICE).eval()
    sd = torch.load(DDPM_CKPT, map_location=DEVICE, weights_only=False)
    _ddpm_net.load_state_dict(sd['state_dict'])
    _ddpm = FundusDDPM(_ddpm_net, T=sd.get('T', T), device=DEVICE)
    return _ddpm


@torch.no_grad()
def enhance_ddpm(img_pil, n_steps=50):
    """Restore via conditional DDPM with DDIM-style fast sampling."""
    ddpm = _load_ddpm()
    orig = img_pil.size
    src  = img_pil.convert('RGB').resize((DDPM_SIZE, DDPM_SIZE), Image.BILINEAR)
    tfm  = T.Compose([T.ToTensor(),
                      T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    x = tfm(src).unsqueeze(0).to(DEVICE)
    y = ddpm.restore(x, n_steps=n_steps)
    arr = (y.squeeze().permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
    arr = (arr * 255).clip(0, 255).astype('uint8')
    return Image.fromarray(arr).resize(orig)


print("Conditional DDPM restorer ready.")


## Verify the restorer works on the loaded checkpoint

In [ ]:
# === DDPM smoke test: restore a few degraded test images (uses the loaded checkpoint) ===
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

_demo = pd.read_csv(PRISTINE_CSV).sample(3, random_state=SEED)['id_code'].tolist()
_out  = RESULTS_ROOT / 'ddpm' / 'resume_smoke'
_out.mkdir(parents=True, exist_ok=True)

for _kind, _level in [('blur', 'high'), ('noise', 'high'), ('exposure', 'high')]:
    fig, axes = plt.subplots(len(_demo), 3, figsize=(9, 3 * len(_demo)))
    if len(_demo) == 1: axes = axes[None, :]
    for r, _id in enumerate(_demo):
        clean    = Image.open(resolve_image(_id)).convert('RGB').resize((DDPM_SIZE, DDPM_SIZE))
        degraded = apply_degradation(clean, _kind, _level)
        restored = enhance_ddpm(degraded, n_steps=50)
        for c, (im, ttl) in enumerate(zip([clean, degraded, restored],
                                          ['clean', f'{_kind}/{_level}', 'DDPM restored'])):
            axes[r, c].imshow(im); axes[r, c].axis('off')
            if r == 0: axes[r, c].set_title(ttl, fontsize=10)
    fig.suptitle(f'Conditional DDPM restore - {_kind}/{_level}', fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(_out / f'ddpm_resume_{_kind}_{_level}.png', dpi=120, bbox_inches='tight')
    plt.show()
print('Smoke-test figures saved ->', _out)
